# 1. Project Title

# AttritionIQ - IBM HR Analytics Employee Attrition

**Programme:** IBM SkillsBuild Data Analytics with AI - Academic Internship 2026  
**Author:** Khizra  
**Dataset:** IBM HR Analytics Employee Attrition & Performance  

---

> **Disclaimer:** All predictions are statistical estimates for decision-support purposes only.  
> The IBM HR dataset is entirely fictional - no real employees are represented.  
> Model outputs do not establish causation and must not be used for automated employment decisions.

## 2. Problem Statement

Employee attrition - the voluntary or involuntary departure of staff - is costly for organisations. Replacing a single employee can cost 50-200% of their annual salary when accounting for recruitment, onboarding, and lost productivity.

Traditional HR management relies on exit interviews and annual surveys - **lagging indicators** that capture information only after attrition has already occurred.

**The problem:** Given observable employee attributes (demographics, job characteristics, compensation, satisfaction levels, and tenure history), can a machine learning model identify employees with elevated attrition risk early enough for HR to intervene?

**Additional challenge:** The dataset is class-imbalanced (~16% attrition vs. 84% retention). A naive model that always predicts "No Attrition" achieves 84% accuracy while providing zero HR value. Evaluation must therefore go beyond raw accuracy.

## 3. Objectives

1. Perform exploratory data analysis on the IBM HR Analytics dataset to uncover key attrition patterns.
2. Build and compare four machine learning classifiers with class-imbalance handling.
3. Select the best model using a recall-prioritising composite metric: **(F1 + Recall + ROC-AUC) / 3**.
4. Interpret the selected model using feature importance and directional coefficient analysis.
5. Demonstrate a live employee attrition prediction using the trained pipeline.
6. Generate actionable business insights from observed data patterns.
7. Ensure all outputs carry appropriate ethical disclaimers.

## 4. Dataset Description

| Attribute | Detail |
|---|---|
| **Name** | IBM HR Analytics Employee Attrition & Performance |
| **Source** | IBM (fictional dataset - no real employees) |
| **Rows** | 1,470 employee records |
| **Columns** | 35 features |
| **Target** | `Attrition` - Yes (left) / No (stayed) |
| **Class split** | 237 Yes (16.1%) / 1,233 No (83.9%) |

**Feature categories:**
- **Personal:** Age, Gender, MaritalStatus, Education, EducationField
- **Job:** Department, JobRole, JobLevel, BusinessTravel, OverTime
- **Compensation:** MonthlyIncome, DailyRate, HourlyRate, MonthlyRate, PercentSalaryHike, StockOptionLevel
- **Satisfaction:** JobSatisfaction, WorkLifeBalance, EnvironmentSatisfaction, RelationshipSatisfaction, JobInvolvement
- **Tenure:** YearsAtCompany, TotalWorkingYears, YearsInCurrentRole, YearsSinceLastPromotion, YearsWithCurrManager, NumCompaniesWorked

**Columns excluded from modelling:**
- `EmployeeCount`, `Over18`, `StandardHours` - zero-variance constants
- `EmployeeNumber` - unique identifier (no predictive value)

## 5. Import Libraries

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Scikit-learn - preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Scikit-learn - models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Scikit-learn - evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, auc,
    ConfusionMatrixDisplay, classification_report
)

print('All libraries imported successfully.')
print(f'NumPy  {np.__version__} | Pandas {pd.__version__}')

## 6. Load Dataset

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'WA_Fn-UseC_-HR-Employee-Attrition.csv')

df = pd.read_csv(DATA_PATH)

print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'File path     : {os.path.abspath(DATA_PATH)}')
df.head()

## 7. Data Inspection

In [ ]:
print('=== DataFrame Info ===')
df.info()

In [ ]:
print('=== Descriptive Statistics (numeric columns) ===')
df.describe().T.round(2)

In [ ]:
print('=== Categorical column unique values ===')
cat_preview = df.select_dtypes(include='object').columns.tolist()
for col in cat_preview:
    print(f'  {col:30s}: {df[col].unique().tolist()}')

## 8. Data Quality Checks

In [ ]:
# ── Missing values ──
missing = df.isnull().sum()
total_missing = missing.sum()
print(f'Total missing values: {total_missing}')
if total_missing > 0:
    print(missing[missing > 0])
else:
    print('No missing values - dataset is complete.')

In [ ]:
# ── Duplicate rows ──
n_dup = df.duplicated().sum()
print(f'Duplicate rows: {n_dup}')

# ── Constant columns (zero variance) ──
CONSTANT_COLUMNS = ['EmployeeCount', 'Over18', 'StandardHours']
ID_COLUMNS       = ['EmployeeNumber']
print()
print('Constant columns (to be removed before modelling):')
for col in CONSTANT_COLUMNS:
    uvals = df[col].unique()
    print(f'  {col:20s} → unique values: {uvals}')

print()
print(f'ID column excluded: {ID_COLUMNS}')

In [ ]:
# ── Target class distribution ──
TARGET = 'Attrition'
counts = df[TARGET].value_counts()
pcts   = df[TARGET].value_counts(normalize=True).mul(100).round(1)

print('Target class distribution:')
print(pd.DataFrame({'Count': counts, 'Percentage %': pcts}))

imbalance = counts['No'] / counts['Yes']
print(f'\nClass imbalance ratio : {imbalance:.1f}:1  (No:Yes)')
print('Handling strategy     : class_weight="balanced" applied to all classifiers')

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['No (Retained)', 'Yes (Left)'],
              [counts['No'], counts['Yes']],
              color=['#3b82d4', '#dc2626'], edgecolor='white', width=0.5)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 15,
            str(int(bar.get_height())),
            ha='center', fontsize=11, fontweight='bold')
ax.set_title('Target Class Distribution (Attrition)', fontweight='bold', fontsize=12)
ax.set_ylabel('Count')
ax.set_ylim(0, counts['No'] * 1.12)
plt.tight_layout()
plt.show()

## 9. Exploratory Data Analysis

All statistics below are observed associations in the fictional IBM HR dataset.  
**They do not establish causal relationships.**

In [ ]:
# Helper: attrition rate % by any categorical or binned column
def attrition_rate_by(col, data=None):
    d = data if data is not None else df
    return (
        d.groupby(col)[TARGET]
         .apply(lambda x: (x == 'Yes').mean() * 100)
         .round(1)
    )

# ── 9.1 Attrition by Department ──
dept_rate = attrition_rate_by('Department').sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(dept_rate.index, dept_rate.values, color='#dc2626')
for i, v in enumerate(dept_rate.values):
    ax.text(v + 0.3, i, f'{v}%', va='center', fontsize=10)
ax.set_xlabel('Attrition Rate (%)')
ax.set_title('Attrition Rate by Department', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── 9.2 Attrition by Job Role ──
role_rate = attrition_rate_by('JobRole').sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(role_rate.index, role_rate.values, color='#3b82d4')
for i, v in enumerate(role_rate.values):
    ax.text(v + 0.3, i, f'{v}%', va='center', fontsize=9)
ax.set_xlabel('Attrition Rate (%)')
ax.set_title('Attrition Rate by Job Role', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── 9.3 Attrition by Overtime ──
ot_rate = attrition_rate_by('OverTime')
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

colors = ['#3b82d4', '#dc2626']
axes[0].bar(ot_rate.index, ot_rate.values, color=colors, edgecolor='white', width=0.4)
for i, v in enumerate(ot_rate.values):
    axes[0].text(i, v + 0.5, f'{v}%', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Attrition Rate by Overtime', fontweight='bold')
axes[0].set_ylabel('Attrition Rate (%)')
axes[0].set_ylim(0, ot_rate.max() * 1.2)

# ── 9.4 Attrition by Job Satisfaction ──
js_labels = {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'}
js_rate = attrition_rate_by('JobSatisfaction').rename(index=js_labels)
axes[1].bar(js_rate.index, js_rate.values, color='#7c5cd8', edgecolor='white', width=0.5)
for i, v in enumerate(js_rate.values):
    axes[1].text(i, v + 0.3, f'{v}%', ha='center', fontsize=10, fontweight='bold')
axes[1].set_title('Attrition Rate by Job Satisfaction', fontweight='bold')
axes[1].set_ylabel('Attrition Rate (%)')

plt.tight_layout(); plt.show()

print(f'Overtime attrition: Yes={ot_rate["Yes"]}%  No={ot_rate["No"]}%  '
      f'(difference: {ot_rate["Yes"] - ot_rate["No"]:.1f} pp)')

In [ ]:
# ── 9.5 Attrition by Age Group & Tenure ──
df_eda = df.copy()
df_eda['AgeGroup'] = pd.cut(df_eda['Age'],
    bins=[17, 25, 35, 45, 55, 100],
    labels=['18-25', '26-35', '36-45', '46-55', '55+'])
df_eda['TenureBucket'] = pd.cut(df_eda['YearsAtCompany'],
    bins=[-1, 2, 5, 10, 20, 100],
    labels=['0-2 yrs', '3-5 yrs', '6-10 yrs', '11-20 yrs', '20+ yrs'])

age_rate    = attrition_rate_by('AgeGroup',    data=df_eda)
tenure_rate = attrition_rate_by('TenureBucket', data=df_eda)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(age_rate.index.astype(str), age_rate.values, color='#d97706', edgecolor='white')
axes[0].set_title('Attrition Rate by Age Group', fontweight='bold')
axes[0].set_ylabel('Attrition Rate (%)')

axes[1].bar(tenure_rate.index.astype(str), tenure_rate.values, color='#3b82d4', edgecolor='white')
axes[1].set_title('Attrition Rate by Tenure', fontweight='bold')
axes[1].set_ylabel('Attrition Rate (%)')

plt.tight_layout(); plt.show()

In [ ]:
# ── 9.6 Monthly Income distribution ──
fig, ax = plt.subplots(figsize=(8, 4))
for label, color in [('Yes', '#dc2626'), ('No', '#3b82d4')]:
    df[df[TARGET] == label]['MonthlyIncome'].plot(
        kind='hist', bins=30, alpha=0.6, color=color,
        label=f'Attrition: {label}', ax=ax)
ax.set_title('Monthly Income Distribution by Attrition', fontweight='bold')
ax.set_xlabel('Monthly Income ($)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout(); plt.show()

print('Average Monthly Income:')
print(df.groupby(TARGET)['MonthlyIncome'].mean().round(0).to_string())

In [ ]:
# ── 9.7 Correlation heatmap (numeric features) ──
num_cols_eda = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
corr = df[num_cols_eda].corr()

fig, ax = plt.subplots(figsize=(14, 11))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(num_cols_eda)))
ax.set_yticks(range(len(num_cols_eda)))
ax.set_xticklabels(num_cols_eda, rotation=90, fontsize=8)
ax.set_yticklabels(num_cols_eda, fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.04)
ax.set_title('Numeric Feature Correlation Matrix', fontweight='bold', fontsize=12)
plt.tight_layout(); plt.show()

## 10. Data Preprocessing

Preprocessing exactly mirrors `src/data_preprocessing.py` in the AttritionIQ application.

**Steps:**
1. Remove constant columns and the ID column
2. Encode the target: `Yes → 1`, `No → 0`
3. Separate numeric and categorical features
4. Apply `StandardScaler` to numeric features
5. Apply `OneHotEncoder` to categorical features
6. Combine with `ColumnTransformer` inside a `Pipeline`

In [ ]:
# ── Step 1: Remove constant and ID columns ──
CONSTANT_COLUMNS = ['EmployeeCount', 'Over18', 'StandardHours']
ID_COLUMNS       = ['EmployeeNumber']
TARGET           = 'Attrition'
RANDOM_STATE     = 42
TEST_SIZE        = 0.20

df_model = df.copy()

# ── Step 2: Encode target ──
df_model[TARGET] = (df_model[TARGET] == 'Yes').astype(int)

drop_cols = [c for c in CONSTANT_COLUMNS + ID_COLUMNS + [TARGET] if c in df_model.columns]
X = df_model.drop(columns=drop_cols)
y = df_model[TARGET]

print(f'Feature matrix shape : {X.shape}')
print(f'Target vector shape  : {y.shape}')
print(f'Target distribution  : {dict(y.value_counts())}')

## 11. Encoding Categorical Variables

Categorical features are encoded using **OneHotEncoder** (`handle_unknown='ignore'`), which creates one binary column per category.  
This is applied inside the `ColumnTransformer` pipeline to prevent data leakage (the encoder is fit only on training data).

In [ ]:
# ── Step 3: Identify numeric vs. categorical columns ──
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numeric features  ({len(num_cols)}): {num_cols}')
print()
print(f'Categorical features ({len(cat_cols)}): {cat_cols}')
print()
print('Sample encoding preview - OverTime:')
print(pd.get_dummies(df[['OverTime']], prefix='OverTime').head(4).to_string())

## 12. Feature Preparation

A **ColumnTransformer** is built to apply:  
- `StandardScaler` → numeric features (zero mean, unit variance)  
- `OneHotEncoder` → categorical features

The transformer is wrapped inside a `Pipeline` together with each classifier - ensuring the scaler and encoder are fitted **only** on training data, preventing any leakage from the test set.

In [ ]:
# ── Step 4 & 5: Build ColumnTransformer ──
num_pipeline = Pipeline([('scaler',  StandardScaler())])
cat_pipeline = Pipeline([('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols),
])

print('ColumnTransformer built:')
print(f'  Numeric  ({len(num_cols)} cols)  → StandardScaler')
print(f'  Categorical ({len(cat_cols)} cols) → OneHotEncoder')
# Approximate total features after OHE expansion
approx_ohe = sum(df[c].nunique() for c in cat_cols)
print(f'  Approx. total features after encoding: {len(num_cols) + approx_ohe}')

## 13. Train-Test Split

In [ ]:
# ── Step 6: Stratified 80/20 split (random_state=42, same as application) ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'Training samples : {len(X_train):>5}  |  Attrition rate: {y_train.mean()*100:.1f}%')
print(f'Test samples     : {len(X_test):>5}  |  Attrition rate: {y_test.mean()*100:.1f}%')
print()
print('Stratification confirmed - attrition rate is consistent across both splits.')

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for ax, y_s, title in zip(axes,
                          [y_train, y_test],
                          ['Training Set', 'Test Set']):
    counts_s = y_s.value_counts()
    ax.bar(['No (0)', 'Yes (1)'], [counts_s.get(0,0), counts_s.get(1,0)],
           color=['#3b82d4', '#dc2626'], edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
plt.suptitle('Class Distribution After Stratified Split', fontsize=11)
plt.tight_layout(); plt.show()

## 14. Logistic Regression

Logistic Regression is a linear classifier that models the log-odds of the target using a linear combination of input features.  
`class_weight='balanced'` adjusts sample weights inversely proportional to class frequency, compensating for the 84/16 imbalance.

In [ ]:
lr_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(
                         class_weight='balanced',
                         max_iter=1000,
                         random_state=42))
])

lr_pipe.fit(X_train, y_train)
lr_pred  = lr_pipe.predict(X_test)
lr_proba = lr_pipe.predict_proba(X_test)[:, 1]

lr_metrics = {
    'Accuracy' : accuracy_score(y_test, lr_pred),
    'Precision': precision_score(y_test, lr_pred, zero_division=0),
    'Recall'   : recall_score(y_test, lr_pred, zero_division=0),
    'F1'       : f1_score(y_test, lr_pred, zero_division=0),
    'ROC-AUC'  : roc_auc_score(y_test, lr_proba),
}

print('Logistic Regression - Test Set Results:')
for k, v in lr_metrics.items():
    print(f'  {k:<12}: {v*100:.2f}%')
print()
print(classification_report(y_test, lr_pred, target_names=['No Attrition', 'Attrition']))

## 15. Random Forest

Random Forest is an ensemble of decision trees trained on random feature subsets.  
`class_weight='balanced'` ensures minority class (attrition=Yes) errors are penalised proportionally.

In [ ]:
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(
                         n_estimators=200,
                         class_weight='balanced',
                         max_depth=10,
                         random_state=42,
                         n_jobs=-1))
])

rf_pipe.fit(X_train, y_train)
rf_pred  = rf_pipe.predict(X_test)
rf_proba = rf_pipe.predict_proba(X_test)[:, 1]

rf_metrics = {
    'Accuracy' : accuracy_score(y_test, rf_pred),
    'Precision': precision_score(y_test, rf_pred, zero_division=0),
    'Recall'   : recall_score(y_test, rf_pred, zero_division=0),
    'F1'       : f1_score(y_test, rf_pred, zero_division=0),
    'ROC-AUC'  : roc_auc_score(y_test, rf_proba),
}

print('Random Forest - Test Set Results:')
for k, v in rf_metrics.items():
    print(f'  {k:<12}: {v*100:.2f}%')
print()
print(classification_report(y_test, rf_pred, target_names=['No Attrition', 'Attrition']))

## 16. Gradient Boosting

Gradient Boosting builds trees sequentially, with each tree correcting the residual errors of the previous ensemble.  
Class imbalance is managed through the loss function's gradient scaling.

In [ ]:
gb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   GradientBoostingClassifier(
                         n_estimators=200,
                         learning_rate=0.05,
                         max_depth=4,
                         random_state=42))
])

gb_pipe.fit(X_train, y_train)
gb_pred  = gb_pipe.predict(X_test)
gb_proba = gb_pipe.predict_proba(X_test)[:, 1]

gb_metrics = {
    'Accuracy' : accuracy_score(y_test, gb_pred),
    'Precision': precision_score(y_test, gb_pred, zero_division=0),
    'Recall'   : recall_score(y_test, gb_pred, zero_division=0),
    'F1'       : f1_score(y_test, gb_pred, zero_division=0),
    'ROC-AUC'  : roc_auc_score(y_test, gb_proba),
}

print('Gradient Boosting - Test Set Results:')
for k, v in gb_metrics.items():
    print(f'  {k:<12}: {v*100:.2f}%')
print()
print(classification_report(y_test, gb_pred, target_names=['No Attrition', 'Attrition']))

## 17. Decision Tree

A single decision tree partitions the feature space by selecting the best split at each node.  
`max_depth=8` limits overfitting; `class_weight='balanced'` handles the class imbalance.

In [ ]:
dt_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   DecisionTreeClassifier(
                         class_weight='balanced',
                         max_depth=8,
                         random_state=42))
])

dt_pipe.fit(X_train, y_train)
dt_pred  = dt_pipe.predict(X_test)
dt_proba = dt_pipe.predict_proba(X_test)[:, 1]

dt_metrics = {
    'Accuracy' : accuracy_score(y_test, dt_pred),
    'Precision': precision_score(y_test, dt_pred, zero_division=0),
    'Recall'   : recall_score(y_test, dt_pred, zero_division=0),
    'F1'       : f1_score(y_test, dt_pred, zero_division=0),
    'ROC-AUC'  : roc_auc_score(y_test, dt_proba),
}

print('Decision Tree - Test Set Results:')
for k, v in dt_metrics.items():
    print(f'  {k:<12}: {v*100:.2f}%')
print()
print(classification_report(y_test, dt_pred, target_names=['No Attrition', 'Attrition']))

## 18. Accuracy, Precision, Recall, F1 and ROC-AUC - Full Comparison

**Model selection criterion:** equal-weight composite - `(F1 + Recall + ROC-AUC) / 3`  
This prioritises **Recall** because missing a true at-risk employee (false negative) is more costly than a false positive in an HR retention context.

In [ ]:
all_metrics = {
    'Logistic Regression': lr_metrics,
    'Random Forest'      : rf_metrics,
    'Gradient Boosting'  : gb_metrics,
    'Decision Tree'      : dt_metrics,
}

results_df = pd.DataFrame(all_metrics).T
results_df['Combined Score'] = (
    results_df['F1'] + results_df['Recall'] + results_df['ROC-AUC']
) / 3
results_df = results_df.sort_values('Combined Score', ascending=False)

# Display as percentages
display_df = results_df.copy()
for col in display_df.columns:
    display_df[col] = display_df[col].map(lambda x: f'{x*100:.1f}%')

print('=' * 78)
print('  MODEL COMPARISON - sorted by Combined Score (F1 + Recall + ROC-AUC) / 3')
print('=' * 78)
print(display_df.to_string())

best_name = results_df.index[0]
best_combined = results_df.loc[best_name, 'Combined Score']
print(f'\n★  Best model: {best_name}  (Combined Score = {best_combined*100:.2f}%)')

In [ ]:
# ── Grouped bar chart: all metrics across all models ──
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
model_names     = results_df.index.tolist()
palette         = ['#3b82d4', '#dc2626', '#16a34a', '#d97706', '#7c5cd8']

x      = np.arange(len(model_names))
width  = 0.14
fig, ax = plt.subplots(figsize=(12, 5))

for i, (metric, color) in enumerate(zip(metrics_to_plot, palette)):
    vals = [results_df.loc[m, metric] * 100 for m in model_names]
    ax.bar(x + (i - 2) * width, vals, width=width,
           label=metric, color=color, alpha=0.88)

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 105)
ax.set_title('Model Performance Comparison - All Metrics', fontweight='bold', fontsize=13)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.axhline(y=80, color='grey', linestyle=':', linewidth=0.8, alpha=0.6)
plt.tight_layout(); plt.show()

## 19. Confusion Matrix

Confusion matrices for all four models, then a detailed breakdown for the selected best model.

In [ ]:
# ── 2×2 grid of confusion matrices ──
pipelines_all = {
    'Logistic Regression': (lr_pipe, lr_pred),
    'Random Forest'      : (rf_pipe, rf_pred),
    'Gradient Boosting'  : (gb_pipe, gb_pred),
    'Decision Tree'      : (dt_pipe, dt_pred),
}

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (name, (pipe, pred)) in zip(axes.flat, pipelines_all.items()):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Retained', 'Left']
    )
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    star = ' ★ BEST' if name == best_name else ''
    ax.set_title(f'{name}{star}', fontweight='bold', fontsize=10)

plt.suptitle('Confusion Matrices - All Models (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Detailed breakdown for best model ──
best_pred = lr_pred  # Logistic Regression
cm_best   = confusion_matrix(y_test, best_pred)
tn, fp, fn, tp = cm_best.ravel()

print(f'Detailed Confusion Matrix - {best_name} (Test set: {len(y_test)} samples)')
print('-' * 50)
print(f'  True  Negatives  (Retained, correctly)    : {tn:>4}')
print(f'  False Positives  (Retained, flagged risk)  : {fp:>4}')
print(f'  False Negatives  (Left, missed)            : {fn:>4}  ← minimise this')
print(f'  True  Positives  (Left, correctly caught)  : {tp:>4}')
print()
print(f'  Recall = TP / (TP + FN) = {tp} / ({tp}+{fn}) = {tp/(tp+fn)*100:.1f}%')
print(f'  The model correctly identifies {tp} of {tp+fn} true attrition cases.')

## 20. ROC Curve

The ROC (Receiver Operating Characteristic) curve plots True Positive Rate against False Positive Rate at every possible threshold.  
AUC (Area Under the Curve) closer to 1.0 indicates stronger discriminative ability.

In [ ]:
proba_map = {
    'Logistic Regression': lr_proba,
    'Random Forest'      : rf_proba,
    'Gradient Boosting'  : gb_proba,
    'Decision Tree'      : dt_proba,
}
roc_colors = ['#dc2626', '#3b82d4', '#16a34a', '#d97706']

fig, ax = plt.subplots(figsize=(7, 6))
for (name, proba), color in zip(proba_map.items(), roc_colors):
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_val     = auc(fpr, tpr)
    lw = 2.8 if name == best_name else 1.5
    ls = '-'  if name == best_name else '--'
    label = f'{name} (AUC = {roc_val:.3f})'
    if name == best_name:
        label += ' ★'
    ax.plot(fpr, tpr, color=color, lw=lw, ls=ls, label=label)

ax.plot([0, 1], [0, 1], 'k:', lw=1, label='Random Classifier (AUC = 0.500)')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves - All Models', fontweight='bold', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
ax.fill_between(
    *roc_curve(y_test, lr_proba)[:2],
    alpha=0.08, color='#dc2626', label='_nolegend_'
)
plt.tight_layout(); plt.show()

## 21. Feature Importance / Model Interpretation

For **Logistic Regression**, feature importance is the **absolute value of the standardised coefficient**.  
- **Positive coefficient** → feature is associated with *higher* predicted attrition probability (risk-increasing)
- **Negative coefficient** → feature is associated with *lower* predicted attrition probability (protective)

> **These are statistical associations, not causal relationships.  
> The model does not claim that any feature causes attrition.**

In [ ]:
# ── Extract feature names after OHE ──
fitted_pre = lr_pipe.named_steps['preprocessor']
ohe        = fitted_pre.named_transformers_['cat'].named_steps['encoder']
cat_feat_names = ohe.get_feature_names_out(cat_cols).tolist()
all_feat_names = num_cols + cat_feat_names

# ── Coefficients ──
classifier  = lr_pipe.named_steps['classifier']
coefs       = classifier.coef_[0]
abs_coefs   = np.abs(coefs)

feat_df = pd.DataFrame({
    'Feature'    : all_feat_names,
    'Coefficient': coefs,
    'AbsCoef'    : abs_coefs,
}).sort_values('AbsCoef', ascending=False).head(20)

print('Top 20 features by |Coefficient| - Logistic Regression:')
print(feat_df[['Feature', 'Coefficient', 'AbsCoef']].to_string(index=False))

In [ ]:
# ── Directional feature importance chart ──
plot_df = feat_df.iloc[::-1]   # reverse for bottom-up display
bar_colors = ['#dc2626' if v > 0 else '#3b82d4' for v in plot_df['Coefficient']]

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(plot_df['Feature'], plot_df['AbsCoef'], color=bar_colors, edgecolor='white')
ax.set_xlabel('|Coefficient| (standardised)')
ax.set_title('Top 20 Feature Importances - Logistic Regression', fontweight='bold', fontsize=12)
ax.tick_params(axis='y', labelsize=9)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#dc2626', label='Risk-Increasing (positive coef.)'),
    Patch(facecolor='#3b82d4', label='Protective (negative coef.)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout(); plt.show()

print('\nNote: colour indicates direction of association, NOT causation.')

In [ ]:
# ── Risk-increasing vs. protective split ──
risk_factors   = feat_df[feat_df['Coefficient'] > 0][['Feature', 'Coefficient']].head(5)
protect_factors= feat_df[feat_df['Coefficient'] < 0][['Feature', 'Coefficient']].head(5)

print('Top 5 RISK-INCREASING features (positive coefficient):')
print(risk_factors.to_string(index=False))
print()
print('Top 5 PROTECTIVE features (negative coefficient):')
print(protect_factors.to_string(index=False))
print()
print('Disclaimer: These factors are associated with the model prediction')
print('and do not establish causation.')

## 22. Business Insights

The following insights are derived from observed patterns in the fictional IBM HR dataset.  
**All findings are associations only - no causal claims are made.**

In [ ]:
print('=' * 65)
print('  KEY BUSINESS INSIGHTS FROM IBM HR DATASET')
print('  (Observed associations - not causal claims)')
print('=' * 65)

ot     = attrition_rate_by('OverTime')
dept   = attrition_rate_by('Department').sort_values(ascending=False)
role   = attrition_rate_by('JobRole').sort_values(ascending=False)
js_map = {1:'Low',2:'Medium',3:'High',4:'Very High'}
js     = attrition_rate_by('JobSatisfaction').rename(index=js_map)

print(f'\n1. OVERTIME')
print(f'   Overtime=Yes: {ot["Yes"]}%  vs  Overtime=No: {ot["No"]}%')
print(f'   Difference  : {ot["Yes"]-ot["No"]:.1f} percentage points')

print(f'\n2. DEPARTMENT  (highest → lowest)')
for d, r in dept.items():
    print(f'   {d:<35}: {r}%')

print(f'\n3. TOP 3 JOB ROLES BY ATTRITION RATE')
for name_r, rate_r in role.head(3).items():
    print(f'   {name_r:<35}: {rate_r}%')

print(f'\n4. JOB SATISFACTION')
for lvl, rate_j in js.items():
    print(f'   {lvl:<12}: {rate_j}%')

# Average income
inc = df.groupby(TARGET)['MonthlyIncome'].mean().round(0)
print(f'\n5. AVERAGE MONTHLY INCOME')
print(f'   Attrition=Yes: ${inc.get("Yes",0):,.0f}  vs  Attrition=No: ${inc.get("No",0):,.0f}')

# Young/new employees
young = df[df['Age'] <= 25]
young_rate = (young[TARGET] == 'Yes').mean() * 100
new_emp = df[df['YearsAtCompany'] <= 2]
new_rate = (new_emp[TARGET] == 'Yes').mean() * 100
print(f'\n6. TENURE & AGE')
print(f'   Age ≤25  attrition rate: {young_rate:.1f}%')
print(f'   Tenure 0-2 yrs rate    : {new_rate:.1f}%')

print('\n' + '=' * 65)
print('NOTE: These are patterns observed in a fictional dataset.')
print('They should not be applied to real HR decisions without')
print('independent domain validation and expert HR review.')

In [ ]:
# ── Visual summary of business insights ──
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Overtime
axes[0,0].bar(['No Overtime', 'Overtime'], [ot['No'], ot['Yes']],
              color=['#3b82d4', '#dc2626'], edgecolor='white', width=0.4)
axes[0,0].set_title('Attrition by Overtime', fontweight='bold')
axes[0,0].set_ylabel('Attrition Rate (%)')

# Department
axes[0,1].barh(dept.index, dept.values, color='#dc2626')
axes[0,1].set_title('Attrition by Department', fontweight='bold')
axes[0,1].set_xlabel('Attrition Rate (%)')

# Job Satisfaction
axes[1,0].bar(js.index.astype(str), js.values, color='#7c5cd8', edgecolor='white')
axes[1,0].set_title('Attrition by Job Satisfaction', fontweight='bold')
axes[1,0].set_ylabel('Attrition Rate (%)')

# Top 5 job roles
top5_roles = role.head(5).sort_values(ascending=True)
axes[1,1].barh(top5_roles.index, top5_roles.values, color='#3b82d4')
axes[1,1].set_title('Top 5 Job Roles by Attrition Rate', fontweight='bold')
axes[1,1].set_xlabel('Attrition Rate (%)')

plt.suptitle('Business Insights - Observed Attrition Patterns (Not Causal)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## 23. Employee Attrition Prediction

A live end-to-end prediction using the trained Logistic Regression pipeline.  
This mirrors the behaviour of the **AttritionIQ Flask application** at `/api/predict`.

> **Disclaimer:** This is a decision-support estimate only.  
> The result does not establish causation and must not be used for automated employment decisions.

In [ ]:
# ── Define a sample employee profile ──
sample_employee = pd.DataFrame([{
    'Age'                    : 28,
    'BusinessTravel'         : 'Travel_Frequently',
    'DailyRate'              : 500,
    'Department'             : 'Sales',
    'DistanceFromHome'       : 25,
    'Education'              : 2,
    'EducationField'         : 'Marketing',
    'EnvironmentSatisfaction': 1,
    'Gender'                 : 'Male',
    'HourlyRate'             : 45,
    'JobInvolvement'         : 2,
    'JobLevel'               : 1,
    'JobRole'                : 'Sales Representative',
    'JobSatisfaction'        : 1,
    'MaritalStatus'          : 'Single',
    'MonthlyIncome'          : 2500,
    'MonthlyRate'            : 10000,
    'NumCompaniesWorked'     : 4,
    'OverTime'               : 'Yes',
    'PercentSalaryHike'      : 11,
    'PerformanceRating'      : 3,
    'RelationshipSatisfaction': 1,
    'StockOptionLevel'       : 0,
    'TotalWorkingYears'      : 4,
    'TrainingTimesLastYear'  : 1,
    'WorkLifeBalance'        : 1,
    'YearsAtCompany'         : 2,
    'YearsInCurrentRole'     : 1,
    'YearsSinceLastPromotion': 2,
    'YearsWithCurrManager'   : 1,
}])

# ── Run prediction ──
prob  = lr_pipe.predict_proba(sample_employee)[0][1]
pred  = lr_pipe.predict(sample_employee)[0]
label = 'YES - Attrition Likely' if pred == 1 else 'NO - Likely to Stay'

if prob >= 0.60:
    risk = 'HIGH'
elif prob >= 0.30:
    risk = 'MEDIUM'
else:
    risk = 'LOW'

print('=' * 50)
print('  EMPLOYEE ATTRITION PREDICTION')
print('=' * 50)
print(f'  Prediction       : {label}')
print(f'  Probability      : {prob*100:.1f}%')
print(f'  Risk Level       : {risk}')
print('=' * 50)
print()
print('Employee profile highlights:')
print(f'  Age={sample_employee["Age"].iloc[0]}, Dept=Sales, Role=Sales Representative')
print(f'  OverTime=Yes, JobSatisfaction=1 (Low), WorkLifeBalance=1 (Bad)')
print(f'  MonthlyIncome=$2,500, StockOptionLevel=0, DistanceFromHome=25 km')
print()
print('DISCLAIMER: This prediction is a statistical estimate for decision-support only.')
print('It does not establish causation and must not be used for automated HR decisions.')

In [ ]:
# ── Probability gauge visualisation ──
fig, ax = plt.subplots(figsize=(6, 2))
bar_color = '#dc2626' if risk == 'HIGH' else ('#d97706' if risk == 'MEDIUM' else '#16a34a')
ax.barh(['Attrition\nProbability'], [prob * 100], color=bar_color, height=0.4)
ax.barh(['Attrition\nProbability'], [100], color='#f3f4f6', height=0.4, zorder=0)
ax.axvline(x=30, color='#16a34a', linestyle='--', linewidth=1.2, label='Low/Medium (30%)')
ax.axvline(x=60, color='#dc2626', linestyle='--', linewidth=1.2, label='Medium/High (60%)')
ax.set_xlim(0, 100)
ax.set_xlabel('Attrition Probability (%)')
ax.set_title(f'Prediction: {label}  |  Risk: {risk}  |  Probability: {prob*100:.1f}%',
             fontweight='bold', fontsize=11)
ax.legend(loc='lower right', fontsize=8)
ax.text(prob * 100 + 1, 0, f'{prob*100:.1f}%', va='center', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── What-if scenario: change OverTime=No and JobSatisfaction=3 ──
improved_employee = sample_employee.copy()
improved_employee['OverTime']         = 'No'
improved_employee['JobSatisfaction']  = 3
improved_employee['WorkLifeBalance']  = 3
improved_employee['StockOptionLevel'] = 1

prob_improved = lr_pipe.predict_proba(improved_employee)[0][1]

print('WHAT-IF SCENARIO ANALYSIS')
print('Changes applied: OverTime=No, JobSatisfaction=3, WorkLifeBalance=3, StockOptionLevel=1')
print('-' * 55)
print(f'  Original probability : {prob*100:.1f}%')
print(f'  Scenario probability : {prob_improved*100:.1f}%')
print(f'  Change               : {(prob_improved-prob)*100:+.1f} percentage points')
print()
print('This is a MODEL SIMULATION of hypothetical attribute changes.')
print('It does NOT predict whether an employee will actually leave.')
print('Changing a feature value in the model does not establish')
print('that the change will prevent attrition.')

## 24. Conclusion

### Summary

| | |
|---|---|
| **Dataset** | IBM HR Analytics - 1,470 employees, 35 features, 16.1% attrition |
| **Best Model** | Logistic Regression |
| **Selection Criterion** | (F1 + Recall + ROC-AUC) / 3 |
| **Accuracy** | 77.5% |
| **Precision** | 40.2% |
| **Recall** | **83.0%** |
| **F1-Score** | 54.2% |
| **ROC-AUC** | **89.8%** |

### Why Logistic Regression?

Although Random Forest (87.4%) and Gradient Boosting (86.7%) achieve higher raw accuracy, they identify only 36% and 45% of true attrition cases respectively.  
Logistic Regression identifies **83%** of true at-risk employees - catching 83 in every 100 actual leavers.  
In HR analytics, **missing at-risk employees (false negatives) is more costly than false positives**.  
The composite criterion `(F1 + Recall + ROC-AUC) / 3` captures this priority and selects Logistic Regression with a combined score of **0.7565**.

### Key Findings

- Overtime, low job satisfaction, early tenure, and Sales Representative roles are the most strongly associated factors with observed attrition in this dataset
- The model provides actionable predictions deployable via the AttritionIQ Flask REST API
- SHAP-based directional attribution is available per prediction in the live application

### Ethical Statement

> All predictions are **decision-support estimates only**.  
> The IBM HR dataset is **entirely fictional** - no real employees are represented.  
> Model outputs **do not establish causation**.  
> Predictions must **not** be used as the sole basis for employment decisions.  
> HR professionals must apply expert judgment and follow established policies before acting.  
> This project is built for **educational and demonstration purposes only**.